# IASTAM 6.0 - Problem 7 - ship detector on Kaggle

Trains the detector on the **same leakage-free split** as at home, starting from the weights
already trained on the laptop.

**Setup (once):**
1. Accept the rules: <https://www.kaggle.com/c/airbus-ship-detection/rules>
2. Upload `split.csv` + `last.pt` (folder `kaggle_upload`) as a Dataset, e.g. `iastam-p7-state`
3. **+ Add Input** -> the Airbus competition **and** that dataset
4. Session options -> **Accelerator: GPU**, **Internet: On**
5. Run cell 1 (PREFLIGHT). It checks everything and says exactly what is missing - it never
   starts heavy work on a broken setup.

Cells are idempotent: if a session dies, just run them again.

## 1. Preflight - checks everything, changes nothing

In [ ]:
import os, glob, json, shutil, time, sys
from pathlib import Path
import numpy as np, pandas as pd

INPUT_ROOT = Path(os.environ.get('P7_INPUT_ROOT', '/kaggle/input'))
WORK       = Path(os.environ.get('P7_WORK', '/kaggle/working'))
OUT        = WORK / 'yolo_ships'
ok = True

def first(pattern):
    """First match anywhere under the input root (Kaggle nests inputs differently over time)."""
    hits = sorted(glob.glob(str(INPUT_ROOT / '**' / pattern), recursive=True))
    return Path(hits[0]) if hits else None

print('input tree (2 levels):')
for d in sorted(glob.glob(str(INPUT_ROOT / '*'))) + sorted(glob.glob(str(INPUT_ROOT / '*' / '*'))):
    print('   ', d)

seg_csv   = first('train_ship_segmentations_v2.csv')
split_csv = first('split.csv')
img_dir   = None
if seg_csv is not None:
    cand = [Path(p) for p in glob.glob(str(seg_csv.parent / '**' / 'train_v2'), recursive=True)]
    cand += [seg_csv.parent / 'train_v2']
    img_dir = next((c for c in cand if c.is_dir()), None)

def check(label, good, hint):
    global ok
    print(('  OK   ' if good else '  MISS ') + label + ('' if good else '  ->  ' + hint))
    ok = ok and bool(good)

print('\npreflight:')
check(f'Airbus labels  {seg_csv}', seg_csv is not None,
      'accept the competition rules, then + Add Input -> Competitions -> airbus-ship-detection')
check(f'Airbus images  {img_dir}', img_dir is not None and img_dir.is_dir(),
      'the competition input is attached but train_v2/ is missing - re-add the competition itself')
check(f'our split.csv  {split_csv}', split_csv is not None,
      '+ Add Input -> Your Work -> the dataset containing split.csv')

weights_in = first('last.pt') or first('best.pt')
check(f'start weights  {weights_in}', weights_in is not None,
      'optional: add last.pt to continue the laptop run (otherwise training starts from scratch)')

try:
    import torch
    cuda = torch.cuda.is_available()
    check(f'GPU            {torch.cuda.get_device_name(0) if cuda else "none"}', cuda,
          'Session options -> Accelerator -> GPU, then restart the session')
except Exception as e:
    check('GPU', False, f'torch not importable: {e}')

print('\nPREFLIGHT ' + ('PASSED - run the next cell' if ok else 'FAILED - fix the MISS lines above'))
print('cpu cores:', os.cpu_count())

## 2. Rebuild the dataset from our split

Writes YOLO labels and **symlinks** the images (no 8 GB copy). Safe to re-run: existing files are
kept, so a restarted session resumes where it stopped.

In [ ]:
assert ok, 'preflight failed - fix the MISS lines in cell 1 first'

def rle_to_box(rle, shape=(768, 768)):
    """Airbus run-length mask -> (cx, cy, w, h) normalised, same convention as sat7/rle.py."""
    h, w = shape
    nums = np.asarray(str(rle).split(), dtype=np.int64)
    starts, lengths = nums[0::2] - 1, nums[1::2]
    pix = np.concatenate([np.arange(s, s + n) for s, n in zip(starts, lengths)])
    ys, xs = pix % h, pix // h                      # column-major, like the Airbus format
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    return ((x0 + x1 + 1) / 2 / w, (y0 + y1 + 1) / 2 / h, (x1 - x0 + 1) / w, (y1 - y0 + 1) / h)

split = pd.read_csv(split_csv)
seg = pd.read_csv(seg_csv)
seg = seg[seg.EncodedPixels.notna()]
per_image = seg.groupby('ImageId')['EncodedPixels'].apply(list).to_dict()
print(split.groupby('split').agg(tiles=('ImageId', 'size'), ships=('n_ships', 'sum')).to_string())

for s in ('train', 'val', 'test'):
    (OUT / 'images' / s).mkdir(parents=True, exist_ok=True)
    (OUT / 'labels' / s).mkdir(parents=True, exist_ok=True)

t0, made, linked, missing = time.time(), 0, 0, 0
for i, r in enumerate(split.itertuples(), 1):
    src = img_dir / r.ImageId
    dst = OUT / 'images' / r.split / r.ImageId
    lbl = OUT / 'labels' / r.split / (Path(r.ImageId).stem + '.txt')
    if not src.exists():
        missing += 1
        continue
    if not dst.exists():
        try:
            os.symlink(src, dst)
        except OSError:
            shutil.copy2(src, dst)
        linked += 1
    if not lbl.exists():
        boxes = [rle_to_box(x) for x in per_image.get(r.ImageId, [])]
        lbl.write_text(''.join('0 %.6f %.6f %.6f %.6f\n' % b for b in boxes))
        made += 1
    if i % 10000 == 0:
        print(f'   {i}/{len(split)}  ({time.time()-t0:.0f}s)', flush=True)

(OUT / 'data.yaml').write_text(
    f'path: {OUT}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: ship\n')
n_train = len(list((OUT / 'images' / 'train').glob('*.jpg')))
n_val   = len(list((OUT / 'images' / 'val').glob('*.jpg')))
n_test  = len(list((OUT / 'images' / 'test').glob('*.jpg')))
print(f'\nlinked {linked}, labels written {made}, images not found {missing}, {time.time()-t0:.0f}s')
print(f'train {n_train} | val {n_val} | test {n_test}')
assert missing == 0, 'some images of the split were not found in the competition input'
assert n_train > 0 and n_val > 0 and n_test > 0, 'empty split - check split.csv'

## 3. Train

Starts from the laptop weights when they are attached (the learned weights carry over; the epoch
counter restarts, which is why we do **not** use `resume=True` - that would look for the laptop's
own paths and fail here). Falls back to a smaller batch automatically if the GPU runs out of memory.

In [ ]:
try:
    import ultralytics
except ImportError:
    !pip -q install ultralytics
    import ultralytics
import torch
from ultralytics import YOLO
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU disappeared: set Accelerator = GPU and restart the session'

EPOCHS = 10
IMGSZ  = 768
WORKERS = min(4, os.cpu_count() or 2)
start = str(weights_in) if weights_in else 'yolov8n.pt'
print('starting from:', start)

def run(batch):
    return YOLO(start).train(data=str(OUT / 'data.yaml'), epochs=EPOCHS, imgsz=IMGSZ, batch=batch,
                             workers=WORKERS, device=0, project=str(WORK / 'runs'), name='ships',
                             exist_ok=True, patience=5, seed=0, plots=True, val=True)

t0 = time.time()
try:
    res = run(16)
except torch.cuda.OutOfMemoryError:
    print('out of memory at batch 16 -> retrying with batch 8')
    torch.cuda.empty_cache()
    res = run(8)
print(f'training finished in {(time.time()-t0)/60:.0f} min -> {res.save_dir}')

## 4. Evaluate on the untouched test split

In [ ]:
best = Path(res.save_dir) / 'weights' / 'best.pt'
m = YOLO(str(best)).val(data=str(OUT / 'data.yaml'), split='test', imgsz=IMGSZ, device=0, verbose=False)
metrics = {'mAP50': float(m.box.map50), 'mAP50-95': float(m.box.map),
           'precision': float(m.box.mp), 'recall': float(m.box.mr),
           'epochs': EPOCHS, 'imgsz': IMGSZ, 'started_from': start}
print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in metrics.items()}, indent=2))
(WORK / 'wp1_metrics_kaggle.json').write_text(json.dumps(metrics, indent=2))

## 5. Package the results

Download `weights.zip` from the **Output** tab, unzip into
`IASTAM_Problem7/code/runs/ships/weights/`, then run locally:

```bash
.venv312/Scripts/python scripts/wp1_train_detector.py --skip-train
```

In [ ]:
shutil.make_archive(str(WORK / 'weights'), 'zip', Path(res.save_dir) / 'weights')
for f in sorted(WORK.glob('*.zip')) + sorted(WORK.glob('*.json')):
    print(f, f'{f.stat().st_size/1e6:.1f} MB')
print('\ndownload weights.zip from the Output tab')